<a href="https://colab.research.google.com/github/cmunozr/2026_I_AI_GU/blob/main/Exercises/ex2_part1_species_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NGEO316 – Exercise: Convolutional Neural Networks Demonstration
## Part 1: Species Classification (Tomato vs Potato vs Apple vs Grape)

**Course:** AI for Earth and Environmental Sciences  
**University of Gothenburg / Chalmers**

---

### What you will do in this lab

You will train a CNN to classify leaf images from the **PlantVillage** dataset into four plant species:
- 🍅 Tomato
- 🥔 Potato
- 🍎 Apple
- 🍇 Grape

The model must learn to distinguish species purely from leaf shape, texture, and colours.

---

> ⚠️ **Enable GPU before starting:** Go to **Runtime → Change runtime type → T4 GPU**.  
> Training will be roughly 10x faster with a GPU.

---

## Step 0 – Install and import libraries

We use:
- **TensorFlow / Keras** – building and training the CNN
- **TensorFlow Datasets (TFDS)** – downloading PlantVillage automatically
- **Matplotlib** – plotting images and training curves
- **scikit-learn** – computing the confusion matrix

In [ ]:
# Install/upgrade TensorFlow Datasets (already on Colab — this ensures PlantVillage builder is available)
!pip install -q tensorflow-datasets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print('TensorFlow version:', tf.__version__)
print('GPU available    :', len(tf.config.list_physical_devices('GPU')) > 0)

tf.random.set_seed(42)
np.random.seed(42)

---
## Step 1 – Load the PlantVillage dataset

PlantVillage contains ca **55,000 leaf images** across 38 classes (14 plant species x disease types + healthy).  
TFDS downloads and caches it automatically — this takes a few minutes the first time.

In [ ]:
ds_full, info = tfds.load(
    'plant_village',
    split='train',        # PlantVillage ships as a single split; we make our own val split below
    with_info=True,
    as_supervised=True,   # returns (image, label) tuples
    shuffle_files=False,
)

CLASS_NAMES = info.features['label'].names

print(f"Total images : {info.splits['train'].num_examples:,}")
print(f"Total classes: {info.features['label'].num_classes}")
print()
print('All class names:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  [{i:2d}] {name}')

### Identify the classes we need

We want only the **healthy** images for tomato, potato, and grape.  
Find the indices for `Tomato___healthy`, `Potato___healthy`, and `Grape___healthy` in the list above.

In [ ]:
def find_label_index(keyword_list):
    """Return (index, name) for all classes matching ALL given keywords."""
    return [
        (i, name) for i, name in enumerate(CLASS_NAMES)
        if all(kw.lower() in name.lower() for kw in keyword_list)
    ]

tomato_healthy = find_label_index(['tomato', 'healthy'])
potato_healthy = find_label_index(['potato', 'healthy'])
apple_healthy  = find_label_index(['apple',  'healthy'])
grape_healthy  = find_label_index(['grape',  'healthy'])

print('Tomato healthy:', tomato_healthy)
print('Potato healthy:', potato_healthy)
print('Apple healthy :', apple_healthy)
print('Grape healthy :', grape_healthy)

IDX_TOMATO = tomato_healthy[0][0]
IDX_POTATO = potato_healthy[0][0]
IDX_APPLE  = apple_healthy[0][0]
IDX_GRAPE  = grape_healthy[0][0]

NEW_CLASS_NAMES = ['Tomato', 'Potato', 'Apple', 'Grape']
print(f'\nNew labels -> 0: {NEW_CLASS_NAMES[0]}, 1: {NEW_CLASS_NAMES[1]}, 2: {NEW_CLASS_NAMES[2]}, , 3: {NEW_CLASS_NAMES[3]}')

---
## Step 2 – Filter and preprocess the data

Raw PlantVillage images vary in size. We:
1. **Filter** — keep only our three healthy classes
2. **Resize** — all images to 128x128 pixels
3. **Normalise** — pixel values from [0, 255] to [0.0, 1.0]
4. **Re-label** — assign new labels 0 / 1 / 2

In [ ]:
IMG_SIZE   = 128
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

def filter_and_preprocess(image, label):
    """
    Keep only the three healthy classes, resize, normalise, and re-label.
    Returns (image, new_label, keep_flag).
    """
    keep = tf.reduce_any([
        tf.equal(label, IDX_TOMATO),
        tf.equal(label, IDX_POTATO),
        tf.equal(label, IDX_APPLE),
        tf.equal(label, IDX_GRAPE)
    ])

    new_label = tf.constant(0, dtype=tf.int64)   # default: tomato
    new_label = tf.where(tf.equal(label, IDX_POTATO), tf.constant(1, dtype=tf.int64), new_label)
    new_label = tf.where(tf.equal(label, IDX_APPLE),  tf.constant(2, dtype=tf.int64), new_label)
    new_label = tf.where(tf.equal(label, IDX_GRAPE),  tf.constant(3, dtype=tf.int64), new_label)

    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, new_label, keep

ds_filtered = ds_full.map(filter_and_preprocess, num_parallel_calls=AUTOTUNE)
ds_filtered = ds_filtered.filter(lambda img, lbl, keep: keep)
ds_filtered = ds_filtered.map(lambda img, lbl, keep: (img, lbl))

counts = {0: 0, 1: 0, 2: 0, 3:0}
for _, lbl in ds_filtered:
    counts[int(lbl.numpy())] += 1

print('Images per class:')
for i, name in enumerate(NEW_CLASS_NAMES):
    print(f'  {name}: {counts[i]}')
print(f'  Total : {sum(counts.values())}')

### Visualise some samples

Always look at your data before training! This tells you:
- Whether the labels look correct
- What visual differences exist between classes
- What features the CNN might latch onto

In [ ]:
examples = {0: [], 1: [], 2: [], 3: []}
for img, lbl in ds_filtered:
    lbl_int = int(lbl.numpy())
    if len(examples[lbl_int]) < 4:
        examples[lbl_int].append(img.numpy())
    if all(len(v) == 4 for v in examples.values()):
        break

fig, axes = plt.subplots(4, 4, figsize=(12, 9))
fig.suptitle('PlantVillage – Healthy leaves (Part 1 training data)', fontsize=14, y=1.01)

for row, name in enumerate(NEW_CLASS_NAMES):
    for col in range(4):
        ax = axes[row, col]
        ax.imshow(examples[row][col])
        ax.set_title(name, fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.show()

> **Question 1:**  
> Look at the images above. Without using a CNN, what features would *you* use to distinguish tomato, potato, and grape leaves? Note your observations here.
>
> *(Double-click this cell to edit)*

---
## Step 3 – Create train / validation splits

We use 80% of the data for training and 20% for validation.

In [ ]:
total_count = sum(counts.values())
n_train     = int(total_count * 0.8)
n_val       = total_count - n_train

print(f'Training images  : {n_train}')
print(f'Validation images: {n_val}')

# Cache first — scans the filtered dataset once and holds it in RAM.
# For ~1,700 images at 128x128 this is around 250 MB, well within Colab's limit.
ds_cached = ds_filtered.cache()

# Validation: take the LAST n_val images from the cached set (fixed, never shuffled)
ds_val_raw   = ds_cached.skip(n_train)

# Training: take the FIRST n_train images, then shuffle freshly each epoch
ds_train_raw = ds_cached.take(n_train)

ds_train = (
    ds_train_raw
    .shuffle(buffer_size=n_train, seed=42, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
ds_val = (
    ds_val_raw
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
print('Data pipelines ready.')

---
## Step 4 – Build the CNN

We build a CNN matching the architecture from the lecture:

```
INPUT (128x128x3)
  |
  |-- CONV (32 filters, 3x3) --> BatchNorm --> ReLU --> MaxPool(2x2)  [output: 64x64x32]
  |-- CONV (64 filters, 3x3) --> BatchNorm --> ReLU --> MaxPool(2x2)  [output: 32x32x64]
  |-- CONV (128 filters,3x3) --> BatchNorm --> ReLU --> MaxPool(2x2)  [output: 16x16x128]
  |
  |-- Flatten
  |-- Dense (128) --> ReLU --> Dropout(0.4)
  +-- Dense (3)   --> Softmax  -->  [Tomato, Potato, Grape]
```

**Recall from the lecture:**
- Each Conv layer uses *shared weights* — the same small filter slides across the whole image
- BatchNorm stabilises training (prevents vanishing/exploding gradients)
- MaxPool halves the spatial size at each block
- The FC head combines all detected features into a final class score

In [ ]:
def build_cnn(num_classes=4, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    model = models.Sequential(name='PlantVillage_CNN')

    # Block 1 -- low-level features: edges, colours, simple textures
    model.add(layers.Conv2D(32, kernel_size=(3, 3), padding='same',
                            input_shape=input_shape, name='conv1'))
    model.add(layers.BatchNormalization(name='bn1'))
    model.add(layers.Activation('relu', name='relu1'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool1'))

    # Block 2 -- mid-level features: corners, curves, repeated patterns
    model.add(layers.Conv2D(64, kernel_size=(3, 3), padding='same', name='conv2'))
    model.add(layers.BatchNormalization(name='bn2'))
    model.add(layers.Activation('relu', name='relu2'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool2'))

    # Block 3 -- high-level features: leaf shapes, vein patterns
    model.add(layers.Conv2D(128, kernel_size=(3, 3), padding='same', name='conv3'))
    model.add(layers.BatchNormalization(name='bn3'))
    model.add(layers.Activation('relu', name='relu3'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool3'))

    # Classifier head
    model.add(layers.Flatten(name='flatten'))
    model.add(layers.Dense(128, activation='relu', name='fc1'))
    model.add(layers.Dropout(0.4, name='dropout'))  # zero out 40% of neurons during training
    model.add(layers.Dense(num_classes, activation='softmax', name='output'))

    return model

model = build_cnn(num_classes=4)
model.summary()

---
## Step 5 – Compile and train

**Loss:** `sparse_categorical_crossentropy` — standard for multi-class problems with integer labels  
**Optimiser:** Adam (lr = 0.001)  
**Early stopping:** stop if validation loss does not improve for 5 epochs, restore best weights

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

print('Starting training — a few minutes on GPU ...')
history = model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=20,
    callbacks=[early_stop],
    verbose=1,
)

---
## Step 6 – Evaluate: Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Part 1 – Species classification training curves', fontsize=13)

epochs_ran = range(1, len(history.history['accuracy']) + 1)

ax1.plot(epochs_ran, history.history['accuracy'],     'o-', label='Train',             color='blue')
ax1.plot(epochs_ran, history.history['val_accuracy'], 's-', label='Validation',        color='red')
ax1.axhline(1/4, color='grey', linestyle='--', alpha=0.5, label='Random baseline (25%)')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0, 1.05)
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, history.history['loss'],     'o-', label='Train',      color='blue')
ax2.plot(epochs_ran, history.history['val_loss'], 's-', label='Validation', color='red')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Cross-entropy loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_acc = max(history.history['val_accuracy'])
print(f'Best validation accuracy: {best_acc:.3f} ({best_acc*100:.1f}%)')

> **Questions – Interpreting training curves:**  
> 1. Does training accuracy keep climbing while validation accuracy flattens or drops? What does that indicate?
> 2. A random classifier on 4 balanced classes achieves 25%. By how much did your CNN beat this baseline?
> 3. Did early stopping trigger? At which epoch? Why is early stopping useful?

---
## Step 7 – Evaluate: Confusion matrix

In [ ]:
y_true, y_pred = [], []
for images, labels in ds_val:
    probs = model.predict(images, verbose=0)  # shape: (batch_size, 3)
    preds = np.argmax(probs, axis=1)           # class with highest probability
    y_pred.extend(preds)
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=NEW_CLASS_NAMES)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Validation confusion matrix – Species classification', fontsize=11)
plt.tight_layout()
plt.show()

print()
print(classification_report(y_true, y_pred, target_names=NEW_CLASS_NAMES))

> **Questions – Interpreting the confusion matrix:**  
> 1. Which species was classified most accurately?
> 2. Which two species were most often confused with each other? Does that make sense given the leaf images?
> 3. Tomato and potato are botanically related (both *Solanum*). Did the CNN reflect this in its mistakes?

---
## Step 8 – Look at what the model gets wrong

Visualising misclassified images is one of the best ways to understand model behaviour.

In [ ]:
all_images, all_true, all_pred = [], [], []
for images, labels in ds_val:
    probs = model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    all_images.extend(images.numpy())
    all_true.extend(labels.numpy())
    all_pred.extend(preds)

all_images = np.array(all_images)
all_true   = np.array(all_true)
all_pred   = np.array(all_pred)

wrong      = all_true != all_pred
wrong_imgs = all_images[wrong]
wrong_true = all_true[wrong]
wrong_pred = all_pred[wrong]

print(f'Misclassified: {wrong.sum()} / {len(all_true)} ({100*wrong.mean():.1f}% error rate)')

n_show = min(12, len(wrong_imgs))
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
fig.suptitle('Misclassified validation images', fontsize=13, y=1.01)

for ax, img, true, pred in zip(
        axes.flat, wrong_imgs[:n_show], wrong_true[:n_show], wrong_pred[:n_show]):
    ax.imshow(img)
    ax.set_title(f'True: {NEW_CLASS_NAMES[true]}\nPred: {NEW_CLASS_NAMES[pred]}',
                 fontsize=8, color='crimson')
    ax.axis('off')
for ax in axes.flat[n_show:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

> **Questions – Misclassified images:**  
> 1. Can you identify the correct species from these images by eye? Are the model's mistakes understandable?
> 2. Do the misclassified images share any common features (lighting, angle, colour)?
> 3. Would a human expert in plant pathology make the same mistakes?

---
## Step 9 – Experiment: Change the architecture

Try at least **one** of the modifications below and note what changes.

| Experiment | What to change |
|------------|----------------|
| A. Fewer layers | Delete conv block 3 entirely |
| B. More filters | Change 32->64, 64->128, 128->256 |
| C. No BatchNorm | Remove all BatchNormalization lines |
| D. No Dropout | Remove the Dropout line |
| E. Larger filters | Change kernel_size=(3,3) to (5,5) everywhere |

In [ ]:
def build_cnn_modified(num_classes=4, input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """
    Copy of build_cnn() -- make your modification here.
    """
    model = models.Sequential(name='PlantVillage_CNN_Modified')

    # Block 1 -- low-level features: edges, colours, simple textures
    model.add(layers.Conv2D(32, kernel_size=(3, 3), padding='same',
                            input_shape=input_shape, name='conv1'))
    model.add(layers.BatchNormalization(name='bn1'))
    model.add(layers.Activation('relu', name='relu1'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool1'))

    # Block 2 -- mid-level features: corners, curves, repeated patterns
    model.add(layers.Conv2D(64, kernel_size=(3, 3), padding='same', name='conv2'))
    model.add(layers.BatchNormalization(name='bn2'))
    model.add(layers.Activation('relu', name='relu2'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool2'))

    # Block 3 -- high-level features: leaf shapes, vein patterns
    model.add(layers.Conv2D(128, kernel_size=(3, 3), padding='same', name='conv3'))
    model.add(layers.BatchNormalization(name='bn3'))
    model.add(layers.Activation('relu', name='relu3'))
    model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool3'))

    # Classifier head
    model.add(layers.Flatten(name='flatten'))
    model.add(layers.Dense(128, activation='relu', name='fc1'))
    model.add(layers.Dropout(0.4, name='dropout'))  # zero out 40% of neurons during training
    model.add(layers.Dense(num_classes, activation='softmax', name='output'))

    return model

model_exp = build_cnn_modified()
model_exp.summary()

In [ ]:
model_exp.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history_exp = model_exp.fit(
    ds_train,
    validation_data=ds_val,
    epochs=30,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)],
    verbose=1,
)

print()
print(f"Original model -- best val accuracy: {max(history.history['val_accuracy']):.3f}")
print(f"Modified model -- best val accuracy: {max(history_exp.history['val_accuracy']):.3f}")

> **Questions – Architecture experiment:**  
> 1. Which experiment did you try? What was your hypothesis?
> 2. Did validation accuracy go up or down? By how much?
> 3. Did the total parameter count change?
> 4. Can you explain the result based on what you learned in the lecture?